# Full monthly Arctic temperature fields from CMIP6

This notebook retrieves monthly near-surface air temperature (`tas`) for the **same CMIP6 model and ensemble member used by the polar-bear sea-ice data**: CNRM-CM6-1, member r1i1p1f2. It exports monthly values at every Arctic grid cell north of 66.5°N for historical data and SSP1-2.6, SSP2-4.5, and SSP5-8.5.

The export keeps the grid-cell and monthly values; it does not average locations, months, or years, and it does not convert values to anomalies. `tas` remains in Kelvin.


## 1. Install and import the data tools

In [1]:
# %pip -q install --upgrade xarray zarr gcsfs cftime

import numpy as np
import pandas as pd
import xarray as xr
import gcsfs
import matplotlib.pyplot as plt

xr.set_options(keep_attrs=True)


## 2. Set the exact CMIP6 stores

The archive notebook documents CNRM-CM6-1, member r1i1p1f2 for the bear data. These direct store paths use that same model/member and the matching SSP1-2.6, SSP2-4.5, and SSP5-8.5 air-temperature runs.


In [2]:
model_id = "CNRM-CM6-1"
member_id = "r1i1p1f2"
pathways = {
    "ssp126": "SSP1-2.6",
    "ssp245": "SSP2-4.5",
    "ssp585": "SSP5-8.5",
}

# Same model and member as the polar-bear sea-ice data.
# These are the CMIP6 Pangeo Google Cloud Zarr stores.
base = "gs://cmip6/CMIP6/ScenarioMIP/CNRM-CERFACS/CNRM-CM6-1"
stores = {
    scenario: f"{base}/{scenario}/{member_id}/Amon/tas/gr/v20190219/"
    for scenario in pathways
}
print(f"Model/member: {model_id} / {member_id}")
print("Pathways:", ", ".join(pathways.values()))


Model/member: CNRM-CM6-1 / r1i1p1f2
Pathways: SSP1-2.6, SSP2-4.5, SSP5-8.5


## 3. Retrieve the full monthly Arctic grid

The regional subset is latitude ≥ 66.5°N. Values are kept at each longitude/latitude cell and monthly time step. This is a subset of CMIP6 to match the bear data’s model, member, and three pathways; it retains all grid cells in the chosen Arctic region.


In [3]:
gcs = gcsfs.GCSFileSystem(token="anon")

def arctic_field(scenario, store):
    ds = xr.open_zarr(gcs.get_mapper(store), consolidated=True)
    tas = ds["tas"]
    lat_name = "lat" if "lat" in tas.coords else "latitude"
    # Mask cells south of the Arctic Circle; preserve monthly, spatial detail.
    return tas.where(tas[lat_name] >= 66.5, drop=True).expand_dims(scenario=[scenario])

fields = [arctic_field(scenario, store) for scenario, store in stores.items()]
full_data = xr.concat(fields, dim="scenario", join="exact")
full_data.name = "tas"
full_data.attrs.update({
    "long_name": "Near-surface air temperature",
    "units": "K",
    "region": "grid cells north of 66.5 degrees north",
    "source": "CMIP6 Pangeo Google Cloud Zarr stores",
    "source_id": model_id,
    "member_id": member_id,
    "grid_label": "gr",
})
full_data


<xarray.DataArray 'tas' (scenario: 3, time: 1032, lat: 17, lon: 256)> Size: 54MB
array([[[[275.07858, 275.6143 , 275.95517, ..., 273.33908, 274.07336,
          274.4643 ],
         [273.59225, 274.35272, 274.83664, ..., 270.56235, 272.28278,
          272.79202],
         [271.8214 , 272.5527 , 273.28403, ..., 267.98727, 269.8473 ,
          270.83435],
         ...,
         [233.81953, 233.81953, 233.81953, ..., 233.62747, 233.81953,
          233.81953],
         [233.80176, 233.80176, 233.80176, ..., 233.80176, 233.80176,
          233.80176],
         [235.14049, 235.14049, 235.14049, ..., 235.14049, 235.14049,
          235.14049]],

        [[276.0762 , 276.58316, 276.9058 , ..., 274.58777, 275.1942 ,
          275.5372 ],
         [274.1347 , 274.7636 , 275.16382, ..., 270.84344, 272.8029 ,
          273.3208 ],
         [271.7646 , 272.46518, 273.16577, ..., 267.11932, 269.2183 ,
          270.49142],
...
         [274.0609 , 274.0609 , 274.0609 , ..., 273.987  , 274.0609 ,
          274.0609 ],
         [274.3421 , 274.3421 , 274.3421 , ..., 274.3421 , 274.3421 ,
          274.3421 ],
         [273.99884, 273.99884, 273.99884, ..., 273.99884, 273.99884,
          273.99884]],

        [[280.06348, 280.26202, 280.3884 , ..., 279.42624, 279.5261 ,
          279.73508],
         [279.34143, 279.5771 , 279.72705, ..., 279.33646, 279.34763,
          279.3452 ],
         [278.95087, 279.11575, 279.2806 , ..., 278.68982, 278.7591 ,
          278.85498],
         ...,
         [272.42166, 272.42166, 272.42166, ..., 272.34665, 272.42166,
          272.42166],
         [272.65393, 272.65393, 272.65393, ..., 272.65393, 272.65393,
          272.65393],
         [272.19226, 272.19226, 272.19226, ..., 272.19226, 272.19226,
          272.19226]]]], shape=(3, 1032, 17, 256), dtype=float32)
Coordinates:
  * scenario  (scenario) object 24B 'ssp126' 'ssp245' 'ssp585'
  * time      (time) datetime64[ns] 8kB 2015-01-16T12:00:00 ... 2100-12-16T12...
  * lat       (lat) float64 136B 66.54 67.94 69.34 70.74 ... 86.14 87.54 88.93
  * lon       (lon) float64 2kB 0.0 1.406 2.812 4.219 ... 355.8 357.2 358.6
    height    float64 8B 2.0
Attributes: (12/15)
    cell_measures:       area: areacella
    cell_methods:        area: time: mean
    description:         Near-Surface Air Temperature
    history:             none
    interval_operation:  900 s
    interval_write:      1 month
    ...                  ...
    units:               K
    region:              grid cells north of 66.5 degrees north
    source:              CMIP6 Pangeo Google Cloud Zarr stores
    source_id:           CNRM-CM6-1
    member_id:           r1i1p1f2
    grid_label:          gr

## 4. Save the full data

NetCDF preserves the time, latitude, and longitude dimensions and is much more suitable than CSV for a full gridded dataset. No regional or temporal averaging is performed in this export.


In [4]:
output_path = "arctic_temperature_monthly_full.nc"
full_data.to_dataset().to_netcdf(output_path, encoding={"tas": {"zlib": True, "complevel": 4}})
print(f"Saved full monthly Arctic grid to {output_path}")
print("Dimensions:", dict(full_data.sizes))
print("Scenarios:", list(full_data.scenario.values))


Saved full monthly Arctic grid to arctic_temperature_monthly_full.nc
Dimensions: {'scenario': 3, 'time': 1032, 'lat': 17, 'lon': 256}
Scenarios: ['ssp126', 'ssp245', 'ssp585']


## What this export includes and excludes

Included: monthly `tas` values at individual Arctic grid cells from CNRM-CM6-1 member r1i1p1f2 for SSP1-2.6, SSP2-4.5, and SSP5-8.5. These are the model/member and pathways documented in the archived polar-bear notebook.

Excluded by design: historical data, grid cells south of 66.5°N, other models, other ensemble members, and other pathways. No months or Arctic grid cells are averaged away. Temperature and sea-ice area are different variables; matching model/member/scenarios makes their projections comparable, but does not make their values directly equivalent. The original CMIP6 Zarr stores remain unchanged in Google Cloud.
